# 🏠 Projet DPE — Prédiction de la Consommation Énergétique des Logements

> **Projet Capstone** — Analyse et modélisation prédictive des Diagnostics de Performance Énergétique (DPE) en France.

[![Site Web](https://img.shields.io/badge/Site%20Web-desptiteseconomies.fr-blue)](https://desptiteseconomies.fr)
[![Python](https://img.shields.io/badge/Python-3.12-yellow)](https://www.python.org/)
[![License](https://img.shields.io/badge/License-Academic-lightgrey)]()

---

## 📋 Sommaire

- [Présentation](#-présentation)
- [Architecture du projet](#-architecture-du-projet)
- [Pipeline de données](#-pipeline-de-données)
- [Modélisation](#-modélisation)
- [Site Web](#-site-web)
- [Résultats](#-résultats)
- [Installation](#-installation)
- [Utilisation](#-utilisation)
- [Technologies](#-technologies)

---

## 🎯 Présentation

Ce projet a pour objectif de **prédire la consommation énergétique des logements français** à partir des données ouvertes du Diagnostic de Performance Énergétique (DPE) fournies par l'[ADEME](https://data.ademe.fr/).

L'approche repose sur des **modèles XGBoost segmentés** par :
- **Région** (Corse, Pays de la Loire, Île-de-France, Normandie, etc.)
- **Tranche de surface** (< 30m², 30-50m², 50-70m², 70-90m², 90-120m², > 120m²)
- **Type de prédiction** : consommation énergétique globale (DPE) et consommation électrique spécifique (Elec)

Les modèles entraînés sont exportés au format JSON et déployés sur un **site web** ([desptiteseconomies.fr](https://desptiteseconomies.fr)) permettant aux utilisateurs d'estimer la consommation de leur logement directement dans le navigateur.

---

## 📁 Architecture du projet

```
Projet-DPE/
│
├── Analyse/                         # Pipeline d'analyse et de modélisation
│   ├── DL_dep_DPE.ipynb             # Téléchargement des données DPE par département (API ADEME)
│   ├── get_data_region.ipynb         # Fusion des données départementales par région
│   ├── Analyse_colonne.ipynb         # Exploration des colonnes de la base DPE
│   ├── Analyse_Finale.ipynb          # Analyse exploratoire approfondie
│   ├── Analyse_main_region.ipynb     # Analyse comparative inter-régions
│   ├── Predictive_Model_Analysis.ipynb # Analyse détaillée des modèles prédictifs
│   │
│   ├── choix_modele_dpe.ipynb        # Sélection du meilleur modèle (DPE)
│   ├── choix_modele_elec.ipynb       # Sélection du meilleur modèle (Électricité)
│   ├── creation_model_dpe.ipynb      # Entraînement du modèle DPE
│   ├── creation_model_elec.ipynb     # Entraînement du modèle Électricité
│   ├── creation_modele_region.ipynb  # Modèles XGBoost par région
│   ├── creation_modele_region_surface.ipynb  # Modèles XGBoost par région × surface
│   │
│   ├── run_predictions.py            # Script d'exécution automatisée des prédictions
│   ├── test_installation.py          # Vérification des dépendances
│   ├── performances_modeles.csv      # Tableau récapitulatif des performances
│   ├── requirements.txt              # Dépendances Python
│   │
│   ├── Corse/                        # Modèles entraînés (.pkl) — Corse
│   ├── Pays_de_la_Loire/             # Modèles entraînés (.pkl) — Pays de la Loire
│   ├── Ile_de_France/                # Modèles entraînés (.pkl) — Île-de-France
│   ├── Normandie/                    # Modèles entraînés (.pkl) — Normandie
│   ├── Bretagne/                     # Modèles entraînés (.pkl) — Bretagne
│   ├── Grand_Est/                    # Modèles entraînés (.pkl) — Grand Est
│   ├── Hauts_de_France/              # Modèles entraînés (.pkl) — Hauts-de-France
│   └── Provence_Alpes_Cote_Azur/     # Modèles entraînés (.pkl) — PACA
│
├── Factures/                         # Analyse des factures réelles
│   ├── Conso_real_clean.ipynb        # Nettoyage des données de consommation réelle
│   ├── Conso_réelle.ipynb            # Analyse de la consommation réelle
│   ├── Fusion.ipynb                  # Fusion des jeux de données
│   └── Modèle1.ipynb                # Modèle sur données facturées
│
├── Site/                             # Site web de prédiction (front-end)
│   ├── index.html                    # Page d'accueil
│   ├── estimation.html               # Formulaire d'estimation de consommation
│   ├── resultats.html                # Affichage des résultats de prédiction
│   ├── a-propos.html                 # Page À propos
│   ├── bons-gestes.html              # Conseils d'économies d'énergie
│   ├── recapitulatif.html            # Récapitulatif utilisateur
│   ├── *-mobile.html                 # Versions mobiles des pages
│   ├── xgboost-predict.js            # Inférence XGBoost côté client (JavaScript)
│   ├── styles.css                    # Feuille de styles
│   └── models_json/                  # Modèles XGBoost exportés en JSON (par région)
│       ├── Corse/
│       ├── Ile_de_France/
│       ├── Pays_de_la_Loire/
│       ├── ... (14 régions)
│
└── README.md
```

---

## 🔄 Pipeline de données

```mermaid
graph LR
    A[API ADEME<br>DPE Logements Existants] -->|DL_dep_DPE.ipynb| B[CSV par Département<br>~227 colonnes]
    B -->|get_data_region.ipynb| C[CSV par Région]
    C -->|Analyse_Finale.ipynb| D[Nettoyage & EDA]
    D -->|creation_modele_region_surface.ipynb| E[Modèles XGBoost<br>par Région × Surface]
    E -->|Export JSON| F[Site Web<br>desptiteseconomies.fr]
```

### 1. Collecte des données
- Source : [API ADEME Data Fair](https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines)
- Pagination automatique avec récupération de toutes les entrées par département
- ~227 colonnes par DPE (caractéristiques du bâtiment, équipements, consommations)

### 2. Fusion régionale
- Agrégation des fichiers départementaux en un fichier par région
- Paramétrable via le code de région et la liste des départements

### 3. Nettoyage & Feature Engineering
- Profilage des colonnes (types, valeurs manquantes, cardinalité)
- Simplification des catégories de chauffage (PAC, Radiateur Électrique, Chaudière, etc.)
- Création de features dérivées (traversant, isolation toiture, tranche de surface)
- Suppression des colonnes avec > 70% de valeurs manquantes

---

## 🤖 Modélisation

### Algorithmes comparés

| Modèle | Type |
|--------|------|
| Régression Linéaire | Linéaire |
| Régression Ridge | Linéaire (régularisé) |
| Random Forest | Ensemble (Bagging) |
| Gradient Boosting | Ensemble (Boosting) |
| **XGBoost** ✅ | **Ensemble (Boosting)** |

### Stratégie de segmentation

Les modèles sont entraînés de manière **segmentée** pour capturer les spécificités locales et structurelles :

- **Par région** : chaque région de France métropolitaine dispose de ses propres modèles
- **Par tranche de surface** : 6 tranches (< 30m², 30-50m², 50-70m², 70-90m², 90-120m², > 120m²)
- **Par type** : DPE (consommation globale 5 usages) et Elec (consommation électrique uniquement)

→ Soit jusqu'à **12 modèles par région** (6 tranches × 2 types)

### Variables explicatives principales

- `surface_habitable_logement` — Surface habitable du logement
- `annee_construction` — Année de construction
- `hauteur_sous_plafond` — Hauteur sous plafond
- `type_batiment` — Maison / Appartement
- `zone_climatique` — Zone climatique réglementaire
- `classe_altitude` — Classe d'altitude
- `type_generateur_chauffage` (simplifié) — Type de système de chauffage
- `isolation_toiture` — Présence d'isolation en toiture
- `logement_traversant` — Logement traversant ou non

### Métriques d'évaluation

- **R² (Coefficient de détermination)** — Proportion de variance expliquée
- **RMSE (Root Mean Squared Error)** — Erreur moyenne de prédiction (en kWh)

### Résultats (extrait)

| Région | Tranche | Type | R² | RMSE |
|--------|---------|------|----|------|
| Corse | 50-70m² | Elec | **0.909** | 911 |
| Corse | < 30m² | Elec | **0.918** | 336 |
| Corse | 70-90m² | DPE | **0.909** | 1 803 |
| Pays de la Loire | 30-50m² | Elec | **0.902** | 835 |
| Pays de la Loire | 90-120m² | DPE | **0.899** | 2 910 |

---

## 🌐 Site Web

Le site **[desptiteseconomies.fr](https://desptiteseconomies.fr)** permet aux utilisateurs d'estimer la consommation énergétique de leur logement.

### Fonctionnalités
- 📝 **Formulaire d'estimation** : saisie des caractéristiques du logement
- 🔮 **Prédiction en temps réel** : inférence XGBoost directement dans le navigateur (JavaScript)
- 📊 **Résultats détaillés** : consommation estimée avec visualisations
- 💡 **Bons gestes** : conseils personnalisés pour réduire sa consommation
- 📱 **Responsive** : versions desktop et mobile dédiées

### Architecture technique
Les modèles XGBoost sont convertis en JSON et interprétés côté client via `xgboost-predict.js`, permettant une prédiction **sans serveur back-end**.

---

## ⚙️ Installation

### Prérequis
- Python 3.12+
- pip

### Mise en place

```bash
# Cloner le dépôt
git clone https://github.com/<votre-username>/Projet-DPE.git
cd Projet-DPE/Analyse

# Créer un environnement virtuel
python -m venv .venv

# Activer l'environnement
# Windows :
.venv\Scripts\activate
# macOS/Linux :
source .venv/bin/activate

# Installer les dépendances
pip install -r requirements.txt
```

### Vérifier l'installation

```bash
python test_installation.py
```

---

## 🚀 Utilisation

### 1. Télécharger les données d'un département

Ouvrir `DL_dep_DPE.ipynb` et modifier la variable `Dep` :
```python
Dep = '44'  # Loire-Atlantique
```

### 2. Fusionner les départements d'une région

Ouvrir `get_data_region.ipynb` et configurer :
```python
region_code = 'PLoire'
departements = ['44', '49', '53', '72', '85']
```

### 3. Entraîner les modèles

Exécuter `creation_modele_region_surface.ipynb` pour générer les modèles XGBoost segmentés par région et tranche de surface.

### 4. Lancer les prédictions batch

```bash
python run_predictions.py
```

---

## 🛠️ Technologies

| Outil | Usage |
|-------|-------|
| **Python 3.12** | Langage principal |
| **Pandas** | Manipulation de données |
| **NumPy** | Calcul numérique |
| **Scikit-learn** | Prétraitement & métriques |
| **XGBoost** | Modèle de Machine Learning |
| **Matplotlib / Seaborn** | Visualisation |
| **Requests** | Appels API ADEME |
| **Jupyter Notebook** | Exploration & prototypage |
| **HTML / CSS / JS** | Site web front-end |
| **GitHub Pages** | Hébergement du site |

---

## 📊 Source des données

- **ADEME** — [Base DPE Logements Existants (depuis juillet 2021)](https://data.ademe.fr/datasets/dpe-v2-logements-existants)
- API : `https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines`
- ~227 colonnes par diagnostic, couvrant les caractéristiques du bâtiment, les équipements, les consommations estimées et les émissions de GES.

---

## 👥 Auteurs

Projet réalisé dans le cadre d'un **Capstone universitaire**.

---

<p align="center">
  <i>Des p'tites économies 🌱</i>
</p>
